<a href="https://colab.research.google.com/github/Magar-Bhuwan/ML-Internship-Assignments/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Content age and growth/decline

**Paper finding:**
FlyRank reported that growing pages were younger on average than declining pages. The reported comparison showed that growing pages averaged about 185 days old, while declining pages averaged about 228 days old.

**My methodology question:**
How was the growth/decline label constructed, and were the content-age measurement and outcome measured over separate time windows? I would want to confirm that information from the outcome period was not used when defining the explanatory variable.

**Why this matters:**
If the measurement windows overlap, the observed relationship could be affected by information from the outcome period. Clarifying the timing would make the finding easier to interpret as an observed association rather than evidence of causation.

### Finding 2 — Freshness and content performance

**Paper finding:**
FlyRank reported that the 31–90 day freshness window had a 5.43:1 growth-to-decline ratio in its analysis. The report also compared recently refreshed pages with pages that had not been refreshed recently.

**My methodology question:**
Were refreshed and unrefreshed pages comparable before the refresh, and does the validation design support interpreting the difference as an effect of refreshing? For example, did the groups differ in age, existing visibility, or other characteristics before the refresh?

**Why this matters:**
Pages selected for refresh may already differ from pages that are not refreshed. Therefore, the observed difference can provide directional evidence of an association, but it does not by itself establish that refreshing caused the improvement.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Evaluation approach

The Week-5 model used a stratified random train/test split. This provides a useful baseline, but rows from the same client could potentially appear in both training and testing.

For this audit, I use a grouped-by-client split. The `client_id` field defines the groups, so all rows belonging to a client are kept entirely in either the training set or the test set.

The purpose is to measure how the same Random Forest approach performs when evaluated on clients that were not represented in training.

I keep the target, feature set, model type, and model parameters consistent with Week 5. The main change is the validation design.


In [66]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

model_df = pd.read_csv(
    "/content/ML-Internship-Assignments/data/raw/content_refresh_anonymized.csv"
)

print("Dataset shape:", model_df.shape)
print("Columns:", model_df.columns.tolist())

Dataset shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [67]:
# Keep only rows with a meaningful trend outcome
model_df = model_df[
    model_df["trend_direction"].isin(
        ["down", "stable", "up", "flat"]
    )
].copy()

# Binary target:
# 1 = declining
# 0 = not declining
model_df["declining_target"] = (
    model_df["trend_direction"] == "down"
).astype(int)

print("Modeling dataset shape:", model_df.shape)

print("\nTarget distribution:")
print(model_df["declining_target"].value_counts())

print("\nTarget distribution (%):")
print(
    model_df["declining_target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Modeling dataset shape: (27764, 45)

Target distribution:
declining_target
1    16262
0    11502
Name: count, dtype: int64

Target distribution (%):
declining_target
1    58.57
0    41.43
Name: proportion, dtype: float64


In [68]:
feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

print("Number of features:", len(feature_columns))

Number of features: 23


### 2.1 Original Week-5 validation

The original Week-5 evaluation used an 80/20 stratified random split. I reproduce that evaluation here using the same target, feature set, Random Forest model, and model parameters.

This provides the "before" result for comparison with the grouped-by-client evaluation.


In [69]:
# Original Week-5 random stratified split

X_original = model_df[feature_columns]
y_original = model_df["declining_target"]

X_train_original, X_test_original, y_train_original, y_test_original = train_test_split(
    X_original,
    y_original,
    test_size=0.20,
    random_state=42,
    stratify=y_original
)

rf_original = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf_original.fit(
    X_train_original,
    y_train_original
)

y_pred_original = rf_original.predict(
    X_test_original
)

original_precision = precision_score(
    y_test_original,
    y_pred_original,
    zero_division=0
)

original_recall = recall_score(
    y_test_original,
    y_pred_original,
    zero_division=0
)

original_f1 = f1_score(
    y_test_original,
    y_pred_original,
    zero_division=0
)

print(f"Week-5 Precision: {original_precision:.4f}")
print(f"Week-5 Recall:    {original_recall:.4f}")
print(f"Week-5 F1:        {original_f1:.4f}")

Week-5 Precision: 0.7043
Week-5 Recall:    0.7959
Week-5 F1:        0.7473


### 2.2 Honest grouped-by-client validation

The dataset contains a `client_id` field, so I use a grouped train/test split for this audit.

All rows belonging to the same client remain in the same split. This prevents the same client from appearing in both training and testing.

This is a more conservative evaluation of generalization to unseen clients than the original row-level random split.


In [70]:
# Honest grouped-by-client split

X = model_df[feature_columns]
y = model_df["declining_target"]
groups = model_df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(
    model_df.iloc[train_idx]["client_id"]
)

test_clients = set(
    model_df.iloc[test_idx]["client_id"]
)

overlap = train_clients.intersection(test_clients)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Overlapping clients:", len(overlap))

Training rows: 22172
Testing rows: 5592
Training clients: 24
Testing clients: 7
Overlapping clients: 0


In [71]:
# Train the same Random Forest under the honest split

rf_honest = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf_honest.fit(
    X_train,
    y_train
)

y_pred_honest = rf_honest.predict(
    X_test
)

In [72]:
honest_precision = precision_score(
    y_test,
    y_pred_honest,
    zero_division=0
)

honest_recall = recall_score(
    y_test,
    y_pred_honest,
    zero_division=0
)

honest_f1 = f1_score(
    y_test,
    y_pred_honest,
    zero_division=0
)

print(f"Honest Precision: {honest_precision:.4f}")
print(f"Honest Recall:    {honest_recall:.4f}")
print(f"Honest F1:        {honest_f1:.4f}")

Honest Precision: 0.6275
Honest Recall:    0.6884
Honest F1:        0.6566


### 2.3 Before vs. after

The following comparison keeps the model, target, and feature set consistent while changing the validation design.

| Validation design              |         Precision |            Recall |                F1 |
| ------------------------------ | ----------------: | ----------------: | ----------------: |
| Week-5 random stratified split |            0.6932 |            0.7599 |            0.7250 |
| ML-09 grouped-by-client split  | 0.6275 | 0.6884 | 0.6566 |

The grouped-by-client result is the primary audited result because the test set contains clients that were not represented in training.


### 2.4 Interpretation

The original Week-5 random split measured an F1-score of 0.7250.

Under the grouped-by-client evaluation, the model measured an F1-score of 0.6566.

The difference shows that measured performance depends on the validation design. The grouped-by-client evaluation provides a more conservative test of generalization to unseen clients.

These results are observations from this dataset and evaluation design. They do not establish that the model will achieve the same performance for every future client or production setting.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### 3.1 Target leakage

The target `declining_target` is constructed from `trend_direction`.

Therefore, `trend_direction` is not used as a model feature.

`trend_pct` is also excluded because it directly describes the observed trend outcome.

These fields would provide information about the outcome being predicted and could make the evaluation misleading.

### 3.2 Temporal leakage

The remaining performance features require a temporal check.

Features such as `impressions_90d`, `clicks_90d`, `sessions_90d`, and related metrics may be valid historical predictors only if their measurement window ends before the period used to determine the target.

The available dataset does not independently provide a prediction timestamp and a clearly separated outcome window for every feature. Therefore, I cannot claim that temporal leakage has been completely ruled out from the dataset alone.

I treat this as an audit limitation rather than declaring the feature set completely leakage-free.


| Feature/group | Leakage concern | Decision |
|---|---|---|
| `trend_direction` | Directly defines the target | Excluded |
| `trend_pct` | Directly describes the outcome | Excluded |
| `content_age_days` | Could be known before prediction | Keep; verify timing |
| `days_since_last_update` | Could be known before prediction | Keep; verify timing |
| `impressions_90d` | Possible overlap with outcome window | Temporal verification needed |
| `clicks_90d` | Possible overlap with outcome window | Temporal verification needed |
| `pageviews_90d` | Possible overlap with outcome window | Temporal verification needed |
| `sessions_90d` | Possible overlap with outcome window | Temporal verification needed |
| `users_90d` | Possible overlap with outcome window | Temporal verification needed |
| `engaged_sessions_90d` | Possible overlap with outcome window | Temporal verification needed |
| `ai_sessions_90d` | Possible overlap with outcome window | Temporal verification needed |
| `days_with_impressions` | Possible overlap with outcome window | Temporal verification needed |
| `days_with_sessions` | Possible overlap with outcome window | Temporal verification needed |
| `ctr` | Derived performance measure | Verify historical window |
| `avg_position` | Derived performance measure | Verify historical window |
| `engagement_rate` | Derived performance measure | Verify historical window |
| `scroll_rate` | Derived performance measure | Verify historical window |
| `ai_traffic_pct` | Derived performance measure | Verify historical window |

In [76]:
# Create error analysis table for the honest grouped test set

error_df = X_test.copy()

error_df["actual"] = y_test.values
error_df["predicted"] = y_pred_honest

error_df["error_type"] = np.select(
    [
        (error_df["actual"] == 1) &
        (error_df["predicted"] == 0),

        (error_df["actual"] == 0) &
        (error_df["predicted"] == 1)
    ],
    [
        "false_negative",
        "false_positive"
    ],
    default="correct"
)

print(error_df["error_type"].value_counts())

error_type
correct           3327
false_positive    1285
false_negative     980
Name: count, dtype: int64


In [77]:
print("False-positive examples:")
display(
    error_df[
        error_df["error_type"] == "false_positive"
    ].head(5)
)

print("False-negative examples:")
display(
    error_df[
        error_df["error_type"] == "false_negative"
    ].head(5)
)

False-positive examples:


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,age_tier_order,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,actual,predicted,error_type
26,0.0,0.00,0.00,2686.0,17181.0,2426,3,9,9,9,...,5,13,0.12,30.0,0.00,11.11,0.0,0,1,false_positive
36,0.0,0.00,0.00,2510.0,15518.0,371,5,6,5,5,...,5,20,1.35,5.4,0.00,0.00,0.0,0,1,false_positive
64,10.0,0.00,0.00,2808.0,19244.0,2639,3,8,6,6,...,4,8,0.11,7.2,0.00,0.00,0.0,0,1,false_positive
73,10.0,0.62,6.26,1300.0,8696.0,13,0,67,65,59,...,5,20,0.00,5.5,12.31,14.93,0.0,0,1,false_positive
82,20.0,0.00,0.00,2793.0,17593.0,1810,8,12,8,7,...,5,13,0.44,8.3,12.50,8.33,0.0,0,1,false_positive


False-negative examples:


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,age_tier_order,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,actual,predicted,error_type
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,10,9,9,...,6,25,0.05,20.3,0.0,10.0,0.0,1,0,false_negative
27,10.0,0.32,10.57,NaN,NaN,1197,2,6,5,5,...,6,104,0.17,8.1,0.0,0.0,0.0,1,0,false_negative
51,0.0,0.00,0.00,2756.0,23316.0,2,0,1,3,3,...,4,8,0.00,7.5,0.0,200.0,0.0,1,0,false_negative
129,30.0,0.00,0.00,2396.0,15734.0,2159,1,3,3,3,...,6,7,0.05,21.3,0.0,0.0,0.0,1,0,false_negative
165,10.0,0.08,0.05,2883.0,18118.0,1035,0,12,12,12,...,6,13,0.00,23.6,0.0,0.0,0.0,1,0,false_negative


### 3.3 Error interpretation

The grouped-by-client model produces both false positives and false negatives.

A false positive occurs when the model predicts declining content but the observed target is not declining. A false negative occurs when the model predicts non-declining content but the observed target is declining.

These errors show that the model should not be treated as a definitive classification of content health.

The model can instead be treated as a decision-support signal for prioritizing content for human review. False positives may create additional review work, while false negatives represent declining items that the model does not flag.

Feature importance should also be interpreted carefully. A feature being important to the fitted model does not mean that the feature causes content decline.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

> The ML-08 Random Forest substantially outperformed the ML-07 baseline on the held-out test set.

### Revised claim

> On the Week-5 held-out test set, the Random Forest measured higher precision, recall, and F1-score than the ML-07 baseline. Under the grouped-by-client validation used in this audit, the model measured **[HONEST PRECISION]** precision, **[HONEST RECALL]** recall, and **[HONEST F1]** F1-score. These results provide directional evidence that the model may be useful as a decision-support signal for prioritizing potentially declining content for human review.

### Why I changed the claim

The original statement was based on a random row-level split. The revised statement reports the measured results and identifies the validation design.

It avoids claiming guaranteed performance on unseen clients and avoids implying that the model will improve content performance in production.

The results should therefore be described as observed and measured evidence from this dataset and evaluation design.


## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.